# 1

In [7]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructField, StructType, StringType, StringType, IntegerType

spark = SparkSession.builder.appName("trst").getOrCreate()

schema = StructType([
    StructField("id", IntegerType()),
    StructField('name', StringType()),
    StructField("age", IntegerType()),
    StructField('city', StringType())

])

df = spark.read.option("sep", "\t").csv("1.csv", header=True, schema=schema)

df.createOrReplaceTempView("temp")


df = df.drop_duplicates()
df.printSchema()

# getting the average age
avg_age = df.select(func.avg(func.col("age"))).first()[0]

# filling in null ages with avg_age
df = df.fillna({'age': int(avg_age)})
df = df.fillna({'name': "Unknown"})

result = df.select(func.col('name'), func.col('age'), func.col('city')).show()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)

+-----+---+----+
| name|age|city|
+-----+---+----+
|Alice| 25|  NY|
| null| 40|  NY|
|  Bob| 32|  LA|
+-----+---+----+



# 2

In [18]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructField, StructType, StringType, StringType, IntegerType

spark = SparkSession.builder.appName("trst").getOrCreate()

df = spark.read.option("sep", ',').csv("2.csv", header=True, inferSchema=True)

df.printSchema()

print("total spent per customer")
grouped = df.groupBy(func.col("customer")).agg(func.sum(func.col("amount")).alias("total_spent"))
df.show()

print('highest spender')
highest_spender = grouped.sort(func.col('total_spent'), ascending=False).limit(1)
highest_spender.show()

print("average purchase amount")
average = df.agg(func.avg('amount')).first()[0]
print(f"{average:.2f}")




root
 |-- customer: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)

total spent per customer
+--------+----------+------+
|customer|   product|amount|
+--------+----------+------+
|       A|     Phone|   500|
|       A|    Laptop|  1000|
|       A|    Tablet|   450|
|       B|     Phone|   700|
|       B|Headphones|   150|
|       B|    Laptop|  1200|
|       C|     Phone|   650|
|       C|   Monitor|   300|
|       C|  Keyboard|    80|
|       D|    Laptop|  1100|
|       D|     Mouse|    40|
|       D|     Phone|   550|
|       E|    Tablet|   500|
|       E|   Monitor|   350|
|       E|Headphones|   200|
|       F|     Phone|   750|
|       F|    Laptop|  1300|
|       F|  Keyboard|    90|
|       G|   Monitor|   400|
|       G|     Mouse|    35|
+--------+----------+------+
only showing top 20 rows
highest spender
+--------+-----------+
|customer|total_spent|
+--------+-----------+
|       K|       2190|
+--------+-----------+

# 3

In [ ]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.window import Window


spark = SparkSession.builder.appName("tesst").getOrCreate()

df = spark.read.option("sep", ",").csv("3.csv", header=True, inferSchema=True)

df.printSchema()

# make the window function
window_spec = Window.partitionBy("department").orderBy(f.col("salary").desc())

# rank the employees
result = (
    df.withColumn("rank", func.row_number().over(window_spec))
    .filter(func.col("rank") == 1)
)

print("highees paied employees per department")
result.drop("rank").show()


root
 |-- department: string (nullable = true)
 |-- employee: string (nullable = true)
 |-- salary: integer (nullable = true)

highees paied employees per department
+-----------+---------+------+----+
| department| employee|salary|rank|
+-----------+---------+------+----+
|Engineering|     Liam|   420|   1|
|Engineering|Charlotte|   410|   2|
|Engineering|      Ava|   400|   3|
|Engineering|   Amelia|   390|   4|
|Engineering|     Noah|   380|   5|
|    Finance|   Robert|   350|   1|
|    Finance|    Linda|   320|   2|
|    Finance|    Kevin|   310|   3|
|    Finance|  Michael|   300|   4|
|    Finance|    Susan|   280|   5|
|         HR|    Chris|   190|   1|
|         HR|      Bob|   180|   2|
|         HR|    Alice|   160|   3|
|         HR|    Nancy|   140|   4|
|         HR|     Jane|   120|   5|
|         IT|    Sarah|   250|   1|
|         IT|     Emma|   220|   2|
|         IT|     Mary|   200|   3|
|         IT|    David|   180|   4|
|         IT|     Alex|   150|   5|
+-----

# 4

In [44]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.types import StructType, StructField

spark = SparkSession.builder.appName("test").getOrCreate()

df_customers = spark.read.option("sep", ",").csv("4_customers.csv", header=True, inferSchema=True)
df_orders = spark.read.csv("4_orders.csv", header=True, inferSchema=True)

df_customers.printSchema()
df_orders.printSchema()

# do an inner join on the 2 tables
join = df_customers.join(df_orders, df_customers.id == df_orders.customer_id, "inner").show()

# get customers with no orders
left_join = df_customers.join(df_orders, df_customers.id == df_orders.customer_id, "left")

print("customers without orders")
left_join.filter(func.col('amount').isNull()).show()


total = left_join.groupBy(func.col("id"), func.col("name")).agg(func.sum(func.col("amount")).alias("total_spent"))

total.drop("id").orderBy(func.col("total_spent").desc()).show()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- amount: integer (nullable = true)

+---+-------+-----------+------+
| id|   name|customer_id|amount|
+---+-------+-----------+------+
|  1|  Alice|          1|   100|
|  1|  Alice|          1|    50|
|  2|    Bob|          2|    75|
|  2|    Bob|          2|   125|
|  3|Charlie|          3|    25|
|  3|Charlie|          3|   200|
|  4|  Diana|          4|   300|
|  4|  Diana|          4|   150|
|  5|    Eve|          5|    80|
|  5|    Eve|          5|    60|
|  6|  Frank|          6|   500|
|  6|  Frank|          6|   120|
|  9|    Ivy|          9|    70|
|  9|    Ivy|          9|    95|
+---+-------+-----------+------+

customers without orders
+---+-----+-----------+------+
| id| name|customer_id|amount|
+---+-----+-----------+------+
|  7|Grace|       NULL|  NULL|
|  8|Henry|       NULL|  NULL|
| 10| Jack|       NULL|  NULL|
+---+-----+-----------+-----

# 5

In [46]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.types import StructField, StructType


spark = SparkSession.builder.appName("test").getOrCreate()

schema = StructType([
    StructField("email", StringType())
])

df = spark.read.csv("5.csv", header=True, schema=schema)

df.groupBy("email").agg(func.count("email").alias("times_shown")).sort("times_shown", ascending=False).show()

+----------------+-----------+
|           email|times_shown|
+----------------+-----------+
|  alice@test.com|          3|
|    bob@test.com|          3|
|   emma@test.com|          2|
|   jack@test.com|          2|
|charlie@test.com|          2|
|  maria@test.com|          2|
| rachel@test.com|          1|
|  nancy@test.com|          1|
| oliver@test.com|          1|
|  grace@test.com|          1|
|  wendy@test.com|          1|
| victor@test.com|          1|
|    tom@test.com|          1|
|  david@test.com|          1|
|   luke@test.com|          1|
|  quinn@test.com|          1|
|  henry@test.com|          1|
|   paul@test.com|          1|
|  sarah@test.com|          1|
|   kate@test.com|          1|
+----------------+-----------+
only showing top 20 rows


# 6

In [56]:
from pyspark.sql import SparkSession, functions as f 
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("6.csv", header=True, inferSchema=True)

# unbounded proceeding - very first row in the partition winodow
# rowsBetween - makes the calculation only on the rows between the spcified rows
window_spec = (Window.partitionBy("user").orderBy("date").
               rowsBetween(Window.unboundedPreceding, Window.currentRow)
            )

# getting the previouse row according to date
window_spec_prev = (Window.partitionBy("user").orderBy("date"))


# makes a new collumn called running_total and performs a sum on all the partitioned rows
result = (df.withColumn("running_total", f.sum("amount").over(window_spec))).show()

# gets the previous transaction using lag (put what you want in the prev and how far back)
prev = df.withColumn("previous_transaction", f.lag("amount", 1).over(window_spec_prev))

prev.show()

# using coalesce to default to a 0 value
# need to use lit(0) pyspark can not do operations on regular python vars (lit(0) creates a column with 0 being the value)
difference_prev = prev.withColumn("difference", f.coalesce(
        f.abs(f.col("amount") - f.col("previous_transaction")),
        f.lit(0)
    ))

difference_prev.show()

+----+-----+------+-------------+
|user| date|amount|running_total|
+----+-----+------+-------------+
|   A| Jan1|    10|           10|
|   A|Jan10|    60|           70|
|   A| Jan2|    20|           90|
|   A| Jan3|    15|          105|
|   A| Jan4|    30|          135|
|   A| Jan5|    25|          160|
|   A| Jan6|    40|          200|
|   A| Jan7|    35|          235|
|   A| Jan8|    50|          285|
|   A| Jan9|    45|          330|
|   B| Jan1|    15|           15|
|   B|Jan10|    70|           85|
|   B| Jan2|    25|          110|
|   B| Jan3|    10|          120|
|   B| Jan4|    35|          155|
|   B| Jan5|    50|          205|
|   B| Jan6|    45|          250|
|   B| Jan7|    30|          280|
|   B| Jan8|    55|          335|
|   B| Jan9|    65|          400|
+----+-----+------+-------------+
only showing top 20 rows
+----+-----+------+--------------------+
|user| date|amount|previous_transaction|
+----+-----+------+--------------------+
|   A| Jan1|    10|                N